<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/Copy_of_Member_3_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf /root/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [4]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "./phi3-neuro-symbolic-adapter"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig, AutoModelForCausalLM
import torch

config = AutoConfig.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=False,
    force_download=True,
    revision="main"
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    config=config,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager",
    revision="main"
)

In [10]:
import os
import zipfile
if not os.path.exists(adapter_path):
    if os.path.exists(adapter_path + ".zip"):
        print(f"Unzipping {adapter_path}.zip to {adapter_path}...")
        with zipfile.ZipFile(adapter_path + ".zip", 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(adapter_path))
    else:
        raise FileNotFoundError(f"Adapter path or zip not found: {adapter_path} or {adapter_path}.zip")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [19]:
import sys
import io
import re
import torch

print("DEFINING THE EXECUTION ENGINE (SANDBOX)")
def run_code_safely(python_code):
    """The Isolated Execution Environment with necessary builtins."""
    restricted_globals = {
        "__builtins__": __builtins__
    }
    restricted_locals = {}

    try:
        if any(bad_word in python_code for bad_word in ["os", "sys", "subprocess", "eval"]):
            return None, "Security Error: Malicious imports detected."

        exec(python_code, restricted_globals, restricted_locals)
        return restricted_locals.get('result', None), None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)}"

def extract_code(text):
    """Robust extraction to handle code blocks with or without markdown backticks."""
    markdown_match = re.search(r'```python\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if markdown_match:
        return markdown_match.group(1).strip()

    instruction_match = re.search(r'python\n(.*?)(?:\n###|$)', text, re.DOTALL | re.IGNORECASE)
    if instruction_match:
        return instruction_match.group(1).strip()

    return None

print("THE AGENTIC LOOP ---")
def solve_problem(question, max_retries=3):
    """The Neuro-Symbolic Swarm Logic with smarter error logging."""
    history = f"### Instruction: Write Python code to solve the math problem. Store the final numerical answer in a variable named 'result'.\n### Question:\n{question}\n### Code:\n"

    for attempt in range(max_retries):
        inputs = tokenizer(history, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        new_text = full_response[len(history):]

        code = extract_code(new_text)
        if not code:
            # Fixed the unterminated string literal here
            history += new_text + "\n### Error: Could not extract Python code. Please wrap your code inside markdown code blocks (e.g., ```python\n...\n```).\n### Rewrite Code:\n"
            continue

        answer, error = run_code_safely(code)

        if error is None and answer is not None:
            return answer, attempt + 1
        history += new_text + f"\n### Error:\n{error}\n### Rewrite Code:\n"

    return None, max_retries

DEFINING THE EXECUTION ENGINE (SANDBOX)
THE AGENTIC LOOP ---


In [12]:
test_df = pd.read_csv("unified_svamp_test.csv")

In [13]:
test_df.shape

(700, 6)

In [21]:
correct = 0
total = 100
results_log = []

for index, row in test_df.iterrows():
    if index >= total:
        break

    print(f"\nEvaluating Question {index + 1}/{total}...")
    target_answer = row['answer']

    original_input = __builtins__.input
    __builtins__.input = lambda *args, **kwargs: "0"

    try:
        agent_answer, attempts = solve_problem(row['question'])
    finally:
        __builtins__.input = original_input

    is_correct = False
    if agent_answer is not None:
        try:
            if abs(float(agent_answer) - float(target_answer)) < 1e-4:
                print(f"Correct! (Took {attempts} attempts)")
                correct += 1
                is_correct = True
            else:
                print(f"Failed. Expected {target_answer}, Got {agent_answer}")
        except ValueError:
            print(f"Failed. Expected {target_answer}, Got non-numeric: {agent_answer}")
    else:
        print(f"Failed. Expected {target_answer}, Got {agent_answer}")

    results_log.append({
        "question": row['question'],
        "target": target_answer,
        "predicted": agent_answer,
        "correct": is_correct,
        "attempts": attempts
    })

print("\n--- FINAL 100-Q BENCHMARK RESULTS ---")
print(f"Total Questions Evaluated: {total}")
print(f"Total Correct: {correct}")
print(f"Final Model Accuracy: {(correct / total) * 100:.2f}%")


Evaluating Question 1/100...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Correct! (Took 1 attempts)

Evaluating Question 2/100...
Failed. Expected 19, Got None

Evaluating Question 3/100...
Failed. Expected 3, Got None

Evaluating Question 4/100...
198.0
Correct! (Took 2 attempts)

Evaluating Question 5/100...
63
Correct! (Took 2 attempts)

Evaluating Question 6/100...
Failed. Expected 322, Got None

Evaluating Question 7/100...
Failed. Expected 30, Got None

Evaluating Question 8/100...
Failed. Expected 192, Got None

Evaluating Question 9/100...
Failed. Expected 26, Got None

Evaluating Question 10/100...
17
Correct! (Took 2 attempts)

Evaluating Question 11/100...
1
Correct! (Took 3 attempts)

Evaluating Question 12/100...
16
Failed. Expected 20, Got 16

Evaluating Question 13/100...
Failed. Expected 4, Got None

Evaluating Question 14/100...
Failed. Expected 2, Got None

Evaluating Question 15/100...
7.0
Correct! (Took 2 attempts)

Evaluating Question 16/100...
Failed. Expected 70, Got None

Evaluating Question 17/100...
345
Correct! (Took 2 attempts)

